# Interrogatorio a los Datos

En esta actividad se analizará el dataset del Titanic utilizando técnicas de sumarización, agrupación, dispersión y muestreo.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Pregunta A - Sumarización Categórica

In [3]:
df['Survived'].value_counts(normalize=True) * 100

Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64

La tasa global de supervivencia fue de aproximadamente 38.38%, mientras que el 61.62% de los pasajeros murió. Esto significa que sobrevivió poco más de una tercera parte de las personas a bordo.

## Pregunta B - Agrupación y Agregación

In [4]:
df.groupby('Sex')['Survived'].mean() * 100

Sex
female    74.203822
male      18.890815
Name: Survived, dtype: float64

Sobrevivió aproximadamente el 74.20% de las mujeres, mientras que solamente sobrevivió el 18.89% de los hombres. La diferencia es bastante grande, por lo que los datos sí muestran que las mujeres tuvieron una mayor prioridad de supervivencia.

## Pregunta C - Detección de Outliers con IQR

In [5]:
df['Fare'].quantile([0.25, 0.75])

0.25     7.9104
0.75    31.0000
Name: Fare, dtype: float64

In [6]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)

IQR = Q3 - Q1

IQR

np.float64(23.0896)

In [7]:
limite_superior = Q3 + 1.5 * IQR

limite_superior

np.float64(65.6344)

In [8]:
outliers_fare = df[df['Fare'] > limite_superior]

len(outliers_fare)

116

In [9]:
outliers_fare['Pclass'].value_counts()

Pclass
1    104
3      7
2      5
Name: count, dtype: int64

El Cuartil 1 fue 7.9104 y el Cuartil 3 fue 31.0. El IQR fue de 23.0896 y el límite superior quedó en aproximadamente 65.63.

Se encontraron 116 pasajeros que pagaron una tarifa superior a este límite. La gran mayoría pertenecía a primera clase, con 104 pasajeros.

## Pregunta D - Media y Mediana de Fare

In [10]:
df['Fare'].mean()

np.float64(32.204207968574636)

In [11]:
df['Fare'].median()

14.4542

La media de Fare es aproximadamente 32.20, mientras que la mediana es 14.45. Que la media sea mucho mayor que la mediana indica que existe una asimetría positiva, causada por algunas tarifas extremadamente altas.

En un algoritmo como K-Nearest Neighbors, si no escalamos esta variable, los valores altos de Fare pueden dominar el cálculo de la distancia euclidiana y hacer que el modelo le dé demasiada importancia a esta característica.

## Pregunta E - Muestreo Estratificado

In [12]:
df['Survived'].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

In [13]:
df['Survived'].value_counts(normalize=True) * 150

Survived
0    92.424242
1    57.575758
Name: proportion, dtype: float64

In [14]:
no_sobrevivientes = df[df['Survived'] == 0].sample(n=92, random_state=42)

sobrevivientes = df[df['Survived'] == 1].sample(n=58, random_state=42)

In [15]:
muestra = pd.concat([no_sobrevivientes, sobrevivientes])

len(muestra)

150

In [16]:
muestra['Survived'].value_counts(normalize=True) * 100

Survived
0    61.333333
1    38.666667
Name: proportion, dtype: float64

Se utilizó un muestreo estratificado para mantener prácticamente la misma proporción de sobrevivientes y no sobrevivientes de la base original.

La muestra tiene 92 no sobrevivientes y 58 sobrevivientes. La proporción no puede ser exactamente idéntica porque una muestra de 150 pasajeros no permite dividir las proporciones originales en números enteros.

Este tipo de muestreo evita un sesgo de representación, donde por azar podríamos terminar con demasiados sobrevivientes o demasiados no sobrevivientes en la muestra.

## Pregunta F - Sesgo por Valores Faltantes

In [17]:
df.groupby('Survived')['Age'].mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

Si la mayoría de las edades faltantes pertenecen a pasajeros de tercera clase que murieron, entonces estos valores faltantes no serían completamente aleatorios.

Como Pandas ignora los NaN al calcular la media, el promedio de edad de los fallecidos podría no representar correctamente a todos los pasajeros de ese grupo. Esto introduce un sesgo por datos faltantes y podría hacer que el modelo aprenda una relación incorrecta entre edad, clase y supervivencia.

## Pregunta G - Tratamiento de Outliers

In [18]:
outliers_fare.sort_values('Fare', ascending=False)[['Pclass', 'Fare']].head(10)

,Pclass,Fare
679,1,512.3292
258,1,512.3292
737,1,512.3292
341,1,263.0000
438,1,263.0000
88,1,263.0000
27,1,263.0000
311,1,262.3750
742,1,262.3750
118,1,247.5208


No eliminaría automáticamente estas filas con .drop(). Aunque matemáticamente sean outliers, una tarifa alta puede ser completamente válida en un barco donde existían pasajeros de primera clase y habitaciones mucho más costosas.

Estos datos pueden contener señal útil sobre el nivel económico o la clase del pasajero. Primero se debería confirmar que no sean errores de captura y, si afectan demasiado al modelo, sería mejor aplicar escalamiento o alguna transformación antes que eliminarlos directamente.

## Pregunta H - Extracción de Títulos

In [20]:
df['Name'].head(10)

0                              Braund, Mr. Owen Harris
1    Cumings, Mrs. John Bradley (Florence Briggs Th...
2                               Heikkinen, Miss. Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                             Allen, Mr. William Henry
5                                     Moran, Mr. James
6                              McCarthy, Mr. Timothy J
7                       Palsson, Master. Gosta Leonard
8    Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)
9                  Nasser, Mrs. Nicholas (Adele Achem)
Name: Name, dtype: object

In [21]:
df['Titulo'] = df['Name'].str.extract(r',\s*([^.]*)\.')[0].str.strip()

df[['Name', 'Titulo']].head()

,Name,Titulo
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr


In [22]:
df.groupby('Titulo')['Age'].median()

Titulo
Capt            70.0
Col             58.0
Don             40.0
Dr              46.5
Jonkheer        38.0
Lady            48.0
Major           48.5
Master           3.5
Miss            21.0
Mlle            24.0
Mme             24.0
Mr              30.0
Mrs             35.0
Ms              28.0
Rev             46.5
Sir             49.0
the Countess    33.0
Name: Age, dtype: float64

El título da información adicional sobre el tipo de pasajero. Por ejemplo, "Master" normalmente representa a un niño, mientras que "Mr" representa a un hombre adulto.

Imputar las edades usando la mediana de cada título es mejor que usar una sola mediana global, porque estamos comparando al pasajero con personas más similares a él. Esto conserva mejor la distribución real de las edades y puede reducir el error del modelo.

## Pregunta I - Varianza de Survived

In [23]:
df['Survived'].var()

0.2367722165474984

La varianza de Survived es aproximadamente 0.23.

Si la varianza fuera exactamente 0.0 significaría que todos los pasajeros tienen el mismo valor, es decir, que todos sobrevivieron o todos murieron.

En ese caso un algoritmo de clasificación no podría aprender a diferenciar entre clases, porque solamente existiría una clase en toda la variable objetivo.

## Pregunta J - Agrupación por Tres Niveles

In [24]:
df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

Al dividir los datos en varias características al mismo tiempo aparecen grupos muy pequeños, incluso algunos con solamente 1 o 2 pasajeros.

Si el modelo aprende reglas basándose en estos casos tan específicos ocurrirá sobreajuste (overfitting), porque estaría tratando casos individuales como si fueran patrones generales. El modelo podría funcionar bien con los datos de entrenamiento, pero tendría problemas al recibir pasajeros nuevos.